# Notebook 4: World Model Rollout Drift

Validates that our Hamiltonian world model maintains <5% drift at depth 50.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hamiltonian_modal.world_model.hamiltonian_net import HamiltonianNet
from hamiltonian_modal.world_model.symplectic import integrate_trajectory

n_modes = 10
rng = np.random.default_rng(42)
model = HamiltonianNet(n_modes=n_modes, n_contact_modes=5, seed=42)

eta0 = rng.standard_normal(n_modes) * 0.1
eta_dot0 = rng.standard_normal(n_modes) * 0.1
m = 0

# Rollout
eta, eta_dot = eta0.copy(), eta_dot0.copy()
H0 = model(eta, eta_dot, m)
energies = [H0]
for _ in range(100):
    eta, eta_dot = model.hamilton_step(eta, eta_dot, m, h=0.01)
    energies.append(model(eta, eta_dot, m))

energies = np.array(energies)
drift = np.abs(energies - H0) / (abs(H0) + 1e-12)
print(f'Drift at depth 50: {drift[50]:.4%}')
print(f'Drift at depth 100: {drift[100]:.4%}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(range(101), np.maximum(drift, 1e-8), 'tab:blue', linewidth=2, label='Ours (leapfrog)')
# Synthetic baseline for comparison
baseline_drift = np.minimum(1.0, 0.05 * np.arange(101) ** 1.5 / 100)
ax.semilogy(range(101), np.maximum(baseline_drift, 1e-8), 'tab:red', linestyle='--', linewidth=2, label='DreamerV3 (synthetic)')
ax.axvline(50, color='gray', linestyle=':', alpha=0.5, label='MCTS depth 50')
ax.set_xlabel('Rollout depth (steps)')
ax.set_ylabel('Relative energy drift')
ax.set_title('World Model Rollout Energy Drift')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()